# 02 · Análise — Desertos de Saúde (SP)
**Objetivo:** responder, com números, três perguntas sobre o acesso a serviços de saúde nos municípios paulistas:
1. Como está a densidade de leitos de internação?
2. Quais municípios são desertos de saúde?
3. Como a cobertura varia entre as regiões — e ela depende da riqueza do município?

**Entradas:** `etl_output/municipios-sp.geojson` e `etl_output/cnes-sp.geojson` (gerados no 01).
**Saídas:** `analise_output/municipios-sp-analise.geojson` e `resumo_por_regiao.csv` (usados no 03).

**Escolhas do trabalho**
- Cobertura = **leitos de internação por 10 mil habitantes** (o trabalho não usa número de médicos).
- Região = **região intermediária do IBGE**.
- Deserto **amplo** = sem pronto-socorro/UPA. Deserto **estrito** = sem pronto-socorro/UPA **e** sem hospital (é o usado nos gráficos).

## 1. Carregar os dados

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import geopandas as gpd, pandas as pd, numpy as np, requests, os

BASE = "/content/drive/MyDrive/Trabalho UBD"
mun = gpd.read_file(f"{BASE}/etl_output/municipios-sp.geojson")
pts = gpd.read_file(f"{BASE}/etl_output/cnes-sp.geojson")
os.makedirs(f"{BASE}/analise_output", exist_ok=True)

# PIB per capita em R$ (int64 evita estouro do int32 ao multiplicar por 1000)
mun["pib_per_capita"] = mun["pib_mil_reais"].astype("int64") * 1000 / mun["populacao"]

# Confere se os arquivos são a versão final da ETL
esperadas = ["municipio", "populacao", "pib_mil_reais", "leitos", "leitos_10k",
             "deserto_urgencia", "n_urgencia", "n_hospitais"]
faltam = [c for c in esperadas if c not in mun.columns]
assert not faltam, f"Colunas ausentes (rode o 01 completo): {faltam}"

tot_pop = mun["populacao"].sum()
print(len(mun), "municípios |", len(pts), "estabelecimentos")
print("Leitos de internação no estado:", int(mun["leitos"].sum()))

Mounted at /content/drive
645 municípios | 135647 estabelecimentos
Leitos de internação no estado: 92265


## 2. Pergunta 1 — Como está a densidade de leitos?

In [ ]:
# Média do estado e mediana dos municípios (leitos por 10 mil hab.)
media_estado = mun["leitos"].sum() / tot_pop * 10_000
print("Estado:", round(media_estado, 1), "| Mediana dos municípios:", round(mun["leitos_10k"].median(), 1))

# Municípios sem nenhum leito
sem_leito = mun[mun["leitos"] == 0]
print(f"Sem nenhum leito: {len(sem_leito)} municípios ({len(sem_leito)/len(mun):.0%}), "
      f"{sem_leito['populacao'].sum():,.0f} habitantes ({sem_leito['populacao'].sum()/tot_pop:.1%} da população)")

# Pior cobertura entre os que têm leito (só cidades com 20 mil+ hab.)
print("\nMenor cobertura entre as cidades com leito (20 mil+ hab.):")
mun[(mun["leitos"] > 0) & (mun["populacao"] >= 20_000)].nsmallest(10, "leitos_10k")[["municipio", "populacao", "leitos", "leitos_10k"]]

Estado: 20.8 | Mediana dos municípios: 6.9
Sem nenhum leito: 293 municípios (45%), 2,231,721 habitantes (5.0% da população)

Menor cobertura entre as cidades com leito (20 mil+ hab.):


,municipio,populacao,leitos,leitos_10k
598,Tremembé,51173,2,0.390831
294,Jandira,118045,13,1.101275
258,Iracemápolis,21967,3,1.365685
65,Artur Nogueira,51456,8,1.554726
85,Barrinha,32092,7,2.181229
187,Embu-Guaçu,66970,17,2.538450
186,Embu das Artes,250691,99,3.949085
253,Iperó,36459,15,4.114210
330,Mairiporã,93853,39,4.155435
96,Biritiba Mirim,29683,13,4.379611


## 3. Pergunta 2 — Quais são os desertos de saúde?

In [ ]:
# Deserto amplo: sem UPA/pronto-socorro
mun["deserto_amplo"] = mun["deserto_urgencia"]
# Deserto estrito: sem UPA/pronto-socorro E sem hospital (definição usada nos gráficos)
mun["deserto_estrito"] = mun["deserto_urgencia"] & (mun["n_hospitais"] == 0)

for nome, col in [("Amplo (sem urgência)", "deserto_amplo"), ("Estrito (sem urgência e sem hospital)", "deserto_estrito")]:
    d = mun[mun[col]]
    print(f"{nome}: {len(d)} municípios ({len(d)/len(mun):.0%}), {d['populacao'].sum():,.0f} hab. ({d['populacao'].sum()/tot_pop:.1%}); "
          f"{(d['populacao'] < 10_000).sum()} têm menos de 10 mil hab.")

# Sem leito E sem urgência
duplo = mun[(mun["leitos"] == 0) & mun["deserto_urgencia"]]
print(f"Sem leito e sem urgência: {len(duplo)} municípios, {duplo['populacao'].sum():,.0f} habitantes")

print("\nMaiores municípios sem pronto-socorro/UPA:")
mun[mun["deserto_amplo"]].nlargest(10, "populacao")[["municipio", "populacao", "leitos"]]

Amplo (sem urgência): 361 municípios (56%), 4,852,261 hab. (10.9%); 229 têm menos de 10 mil hab.
Estrito (sem urgência e sem hospital): 216 municípios (33%), 1,071,320 hab. (2.4%); 201 têm menos de 10 mil hab.
Sem leito e sem urgência: 209 municípios, 1,003,757 habitantes

Maiores municípios sem pronto-socorro/UPA:


,municipio,populacao,leitos
196,Ferraz de Vasconcelos,179198,176
500,Salto,134319,116
316,Lorena,84855,176
553,São Roque,79484,192
130,Campo Limpo Paulista,77632,54
239,Ibiúna,75605,315
168,Cruzeiro,74961,159
315,Lins,74779,328
270,Itapira,72022,903
284,Itupeva,70616,86


## 4. Pergunta 3 — Cobertura por região

In [ ]:
# Região intermediária do IBGE de cada município
lista = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios", timeout=60).json()
reg = pd.DataFrame({
    "cod_ibge": [str(m["id"]) for m in lista],
    "regiao": [((m.get("regiao-imediata") or {}).get("regiao-intermediaria") or {}).get("nome") for m in lista],
})
mun["cod_ibge"] = mun["cod_ibge"].astype(str)
mun = mun.drop(columns=["regiao"], errors="ignore").merge(reg, on="cod_ibge", how="left")
print("Municípios sem região:", int(mun["regiao"].isna().sum()), "(esperado: 0)")

# Resumo por região, da menor para a maior cobertura
por_reg = (mun.groupby("regiao")
              .agg(municipios=("municipio", "count"), populacao=("populacao", "sum"),
                   leitos=("leitos", "sum"), sem_urgencia=("deserto_urgencia", "sum"))
              .assign(leitos_10k=lambda d: (d["leitos"] / d["populacao"] * 10_000).round(1))
              .sort_values("leitos_10k"))
por_reg

Municípios sem região: 0 (esperado: 0)


,municipios,populacao,leitos,sem_urgencia,leitos_10k
regiao,,,,,
Sorocaba,78,3074450,5167,40,16.8
Araçatuba,44,777507,1390,38,17.9
São José dos Campos,39,2505723,4630,20,18.5
Campinas,87,6923591,12944,33,18.7
Araraquara,26,1100714,2184,9,19.8
São Paulo,50,22564260,46304,4,20.5
Ribeirão Preto,64,2546824,5397,21,21.2
Presidente Prudente,55,877245,2332,45,26.6
Marília,54,1023235,2744,38,26.8


## 5. Riqueza × cobertura — PIB per capita e leitos

In [ ]:
from scipy.stats import spearmanr

d = mun.dropna(subset=["pib_per_capita", "leitos_10k"])
print("PIB per capita (R$) mín / mediana / máx:",
      int(d["pib_per_capita"].min()), "/", int(d["pib_per_capita"].median()), "/", int(d["pib_per_capita"].max()))

# Correlação de Spearman em três recortes
recortes = [("Todos os municípios", d),
            ("Só municípios com pelo menos 1 leito", d[d["leitos"] > 0]),
            ("Só cidades com 20 mil+ habitantes", d[d["populacao"] >= 20_000])]
for nome, sub in recortes:
    rho, p = spearmanr(sub["pib_per_capita"], sub["leitos_10k"])
    print(f"{nome} (n={len(sub)}): Spearman = {rho:.2f} (p = {p:.2g})")
# Conclusão automática, com base nas cidades de 20 mil+ hab.
rho20 = spearmanr(recortes[2][1]["pib_per_capita"], recortes[2][1]["leitos_10k"])[0]
if abs(rho20) < 0.3:
    print("\nConclusão: correlação fraca. Município rico não tem, necessariamente, mais leitos por habitante.")
else:
    print("\nConclusão: há relação entre riqueza e leitos (rever o título do gráfico 4 no 03).")

PIB per capita (R$) mín / mediana / máx: 13549 / 46614 / 606740
Todos os municípios (n=645): Spearman = 0.24 (p = 4.3e-10)
Só municípios com pelo menos 1 leito (n=352): Spearman = -0.01 (p = 0.82)
Só cidades com 20 mil+ habitantes (n=252): Spearman = 0.03 (p = 0.62)

Conclusão: correlação fraca. Município rico não tem, necessariamente, mais leitos por habitante.


## 6. Checagens de qualidade
Verificações para saber o quanto confiar nos números antes de apresentar.

In [ ]:
# Municípios sem leito, por faixa de população
sl = mun[mun["leitos"] == 0]
faixas = pd.cut(sl["populacao"], [0, 5_000, 10_000, 20_000, 50_000, 1e9], right=False,
                labels=["<5 mil", "5-10 mil", "10-20 mil", "20-50 mil", ">50 mil"])
print("Municípios sem leito por faixa de população:")
print(faixas.value_counts().sort_index())

# Casos para conferir no CNES: hospital cadastrado, mas 0 leitos
print("\nCom hospital cadastrado e 0 leitos:", mun[(mun["n_hospitais"] > 0) & (mun["leitos"] == 0)]["municipio"].tolist())

# Maiores densidades (podem ser hospitais psiquiátricos ou de longa permanência)
print("\nMaiores densidades:")
print(mun.nlargest(8, "leitos_10k")[["municipio", "populacao", "leitos", "leitos_10k"]])

# Cobertura por região com e sem esses municípios extremos (acima de 100 leitos/10 mil)
sem_out = mun[mun["leitos_10k"] <= 100].groupby("regiao")[["leitos", "populacao"]].sum()
comparacao = pd.DataFrame({"com_todos": por_reg["leitos_10k"],
                           "sem_extremos": (sem_out["leitos"] / sem_out["populacao"] * 10_000).round(1)})
print("\nLeitos por 10 mil hab., por região:")
comparacao.sort_values("com_todos")

Municípios sem leito por faixa de população:
populacao
<5 mil       144
5-10 mil      91
10-20 mil     39
20-50 mil     17
>50 mil        2
Name: count, dtype: int64

Com hospital cadastrado e 0 leitos: ['Panorama', 'Arealva', 'Ariranha', 'Capela do Alto', 'Igaraçu do Tietê', 'Juquitiba', 'Luiziânia', 'Pacaembu', 'Poá']

Maiores densidades:
                      municipio  populacao  leitos  leitos_10k
289                        Jaci       7613     249  327.072113
189    Espírito Santo do Pinhal      39816     674  169.278682
174                Divinolândia      11158     173  155.045707
371                   Nhandeara       9852     129  130.937881
270                     Itapira      72022     903  125.378357
435                     Pirajuí      22431     270  120.369132
521  Santa Rita do Passa Quatro      24833     263  105.907462
250                   Indiaporã       4035      36   89.219331

Leitos por 10 mil hab., por região:


,com_todos,sem_extremos
regiao,,
Sorocaba,16.8,16.8
Araçatuba,17.9,17.9
São José dos Campos,18.5,18.5
Campinas,18.7,16.5
Araraquara,19.8,17.9
São Paulo,20.5,20.5
Ribeirão Preto,21.2,21.2
Presidente Prudente,26.6,26.6
Marília,26.8,26.8


## 7. Salvar a base para os gráficos (03)

In [ ]:
mun.to_file(f"{BASE}/analise_output/municipios-sp-analise.geojson", driver="GeoJSON")
por_reg.to_csv(f"{BASE}/analise_output/resumo_por_regiao.csv")
print("Salvo em analise_output")

Salvo em analise_output
